<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_10_5_dynaface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# T81-558: Applications of Deep Neural Networks

**Module 10: Facial Processing**

* Instructor: [Jeff Heaton](https://sites.washu.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.washu.edu/index.html)
* For more information visit the [class website](https://sites.washu.edu/jeffheaton/t81-558/).

# Module 10 Material

- Part 10.1: Detecting Faces in an Image [[Video]]() [[Notebook]](t81_558_class_10_1_faces.ipynb)
- Part 10.2: Detecting Facial Features [[Video]]() [[Notebook]](t81_558_class_10_2_face_landmarks.ipynb)
- Part 10.3: Reality Augmentation [[Video]]() [[Notebook]](t81_558_class_10_3_reality_augmentation.ipynb)
- Part 10.4: Application: Emotion Detection [[Video]]() [[Notebook]](t81_558_class_10_4_emotion.ipynb)
- **Part 10.5: Dynaface** [[Video]]() [[Notebook]](t81_558_class_10_5_dynaface.ipynb)

# Google CoLab Instructions

The following code checks that Google CoLab is and sets up the correct hardware settings for PyTorch.


In [ ]:
try:
    import google.colab
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# Make use of a GPU or MPS (Apple) if one is available.  (see module 3.2)
import torch
has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


if device!='cuda':
  print("*******WARNING, this notebook requires a CUDA GPU****")
  print("This notebook will not work correctly!")

# Part 9.5: Dynaface

This section is a bit different from the rest of the course because it covers research that comes directly from my own work. Dynaface is an open-source project I built and continue to develop, and it's a useful case study in how the techniques you're learning in this course (facial landmark detection, transfer learning, dataset bias correction, video analysis) come together to solve a real clinical problem. It's also personal. The project began at my kitchen table after my wife, Tracy, woke up from surgery in August 2021 with full paralysis on the left side of her face, and it grew into a collaboration with Dr. Kofi Boahene at Johns Hopkins that has now produced multiple peer-reviewed publications and a tool used by clinicians around the world.

![Dynaface application screenshot](https://s3.amazonaws.com/data.heatonresearch.com/images/facial/site/dynaface-1.jpg)

## Origins of the Project

Tracy's tumor, a benign acoustic neuroma, was successfully removed, but the surgery left her unable to blink, raise her eyebrow, or smile on the left side of her face. The standard medical advice was to wait and see whether the nerve would recover on its own. To cope with the uncertainty, Tracy began taking photographs of her face every day, hoping to catch any flicker of returning movement. I started looking at those photos as data. I wanted a number I could plot on a chart so we could see whether anything was actually changing, even when the human eye couldn't tell.

That instinct, that if you can't measure it you can't really know it, came directly from the actuarial culture I work in at RGA, and it turned out to be the right instinct here. After Tracy received a nerve transfer surgery from Dr. Boahene at Johns Hopkins in early 2022, the slow process of recovery gave us months of daily images that needed to be analyzed. Dr. Boahene later told me that seeing day-by-day quantitative progression of a face during reanimation recovery was something he had never had access to in his career. That conversation turned a personal coping mechanism into a research collaboration, and Dynaface was the result. The full story of how the project started is documented in an [RGA article on the project](https://www.rgare.com/knowledge-center/article/finding-her-smile--ai-expertise-sparks-a-medical-breakthrough).

## The Bias Problem in Pretrained Facial Models

The first thing I tried, naturally, was to use existing pretrained facial landmark models. Every one of them failed in the same way: they "fixed" Tracy's face. They detected the asymmetry as an error rather than as the signal we cared about, and they pulled landmarks toward a more symmetric configuration than what was actually present in the image. This is a classic dataset-bias problem, and it's the same family of issue we deal with constantly in medical AI at RGA. The training data for most facial landmark models is overwhelmingly composed of typical, symmetric faces, so the model has effectively learned that asymmetry is noise to be smoothed away.

For a tool meant to measure paralysis, that's the worst possible failure mode. The asymmetry is exactly what we are trying to quantify. To address this, Dynaface uses [SPIGA (Shape Preserving Facial Landmarks with Graph Attention Networks)](https://github.com/andresprados/SPIGA) as its landmark backbone. SPIGA was designed with shape preservation as an explicit goal, meaning it does not collapse atypical face geometries onto a learned average. In the Dynaface pipeline, SPIGA produces 97 facial landmarks per frame, and these landmarks faithfully reflect the actual position of features on a paralyzed face rather than a regularized version of them. This is the single most important technical decision in the project, because every downstream measurement depends on the landmarks being honest.

## How the Pipeline Works on Still Images

For a single photograph, the Dynaface pipeline runs in roughly four stages. First, the image goes through face detection to crop and orient the face region. Second, SPIGA produces the 97-landmark map. Third, the image is stabilized using the pupils as anchor points, which is critical because a few degrees of head tilt or a different camera distance between two photos can swamp the actual physiological signal you're trying to measure. Fourth, a set of clinical metrics is computed from the landmark coordinates. These metrics were developed in collaboration with Dr. Boahene and include measures of brow height, oral commissure position, eye aperture, and various symmetry ratios that compare corresponding landmarks across the facial midline.

The important part for students of this course is that the whole pipeline is deterministic once the landmarks are fixed. The AI work happens in detection and landmark localization. Everything after that is geometry. This is a useful pattern to internalize: in many applied deep learning projects, the deep learning component is doing the perceptual work of converting pixels into structured coordinates, and a much simpler classical algorithm operates on the structured output. It makes the system more interpretable, easier to debug, and easier for clinicians to trust.

## How the Pipeline Works on Video

Video analysis was the natural next step, and it's where Dynaface goes beyond what a single photo can show. Many of the most clinically interesting phenomena in facial paralysis are dynamic. Synkinesis, for example, is a condition where moving one part of the face involuntarily causes another part to move, such as the eye narrowing when the patient tries to smile. You cannot see this in a still image. You need to see how landmarks move together over time.

For video, Dynaface runs the still-image pipeline frame by frame and then computes time series of each landmark and each derived metric. From those time series we can extract things like blink rate, blink symmetry, peak displacement during expressions, and correlations between movement in different facial regions. The oral-ocular synkinesis paper described below was built on exactly this kind of analysis, and it found that puckering, the expression you make when sucking on a lemon, was the most diagnostically sensitive movement, something that would have been very difficult to discover without quantitative video analysis. All of the per-frame data can be exported to CSV or Excel for downstream statistical work, which is what most of our clinical collaborators want.

## Distribution and the PyInstaller Strategy

A piece of this project that I want to call out, because it's a real-world detail you don't usually see in a deep learning course, is how Dynaface gets onto a clinician's machine. Clinicians and researchers are not, in general, going to set up a Python environment, manage CUDA versions, or pip-install a stack of model dependencies. If we want this tool to actually be used, it has to be a normal application that someone double-clicks.

Dynaface uses [PyInstaller](https://pyinstaller.org/) to bundle the entire Python runtime, the model weights, and all dependencies into a single distributable application for Windows and macOS. This sidesteps the entire class of installation problems that kills adoption of research code, and it lets clinicians run the tool inside hospital IT environments where they often don't have permission to install Python or pip packages at all. There are real costs to this approach. The bundles are large, packaging ONNX runtimes across platforms is finicky, and getting the application notarized and signed for macOS is its own ordeal, but the alternative of asking a surgeon to debug a pip dependency conflict is a non-starter. The repository structure reflects this: there is a `dynaface-lib` directory with the pure Python library that researchers can pip install if they want to script against it, and a `dynaface-app` directory with the desktop application built on top of it.

## Repository Structure

The project is open source under the Apache 2.0 license and is hosted at [github.com/jeffheaton/dynaface](https://github.com/jeffheaton/dynaface). The repository is split into three main pieces. `dynaface-lib` contains the core Python library with the landmark and measurement code, and is the right entry point if you want to use Dynaface programmatically as part of your own analysis pipeline. `dynaface-app` is the Qt-based desktop application, which is the artifact that gets bundled with PyInstaller and is what most non-programmer users interact with. `scripts` contains assorted utilities for batch processing, model conversion, and dataset preparation. Build scripts at the top level handle the cross-platform packaging, and the project uses Jenkins for its CI/CD pipeline because the build matrix (Windows, macOS Intel, macOS Apple Silicon) does not fit comfortably into a single GitHub Actions workflow.

## Publications

Dynaface has been validated in three peer-reviewed publications so far, all in collaboration with the team at Johns Hopkins. If you want to dig deeper into the clinical applications, these are the references:

Renne, A., Heaton, J., & Boahene, K. D. O. (2026). Associations of AI-based facial metrics with patient-reported outcomes in idiopathic facial paralysis. *Laryngoscope*. Advance online publication. [https://doi.org/10.1002/lary.70417](https://doi.org/10.1002/lary.70417)

Renne, A., Heaton, J., Derakhshan, A., Nellis, J. C., Desai, S. C., & Boahene, K. D. (2025). Use of dynamic, automated facial analysis in quantifying oral-ocular synkinesis. *Facial Plastic Surgery & Aesthetic Medicine*. [https://doi.org/10.1177/26893614251395737](https://doi.org/10.1177/26893614251395737)

Berges, A. J., Renne, A., Heaton, J., Leung, D. G., & Boahene, K. D. (2025). Facial weakness in facioscapulohumeral muscular dystrophy: Objective and patient-reported measures to guide reconstructive interventions. *Facial Plastic Surgery & Aesthetic Medicine*, Article 26893614251407675. [https://doi.org/10.1177/26893614251407675](https://doi.org/10.1177/26893614251407675)

A BibTeX file for citing the project is maintained at [github.com/jeffheaton/dynaface/blob/main/CITATION.bib](https://github.com/jeffheaton/dynaface/blob/main/CITATION.bib).

## Takeaways for This Course

I included Dynaface in this course not because the techniques in it are unusual, you've already seen most of them, but because the project illustrates several patterns that you'll run into repeatedly in applied deep learning. Pretrained models carry the biases of their training data, and recognizing when those biases are working against your problem is often more valuable than any modeling trick. Choosing a backbone, in this case SPIGA, that respects the structure you actually care about can save you from training a model from scratch. Most of the engineering work in a real deep learning product is not the model itself but the pipeline around it: stabilization, calibration, deterministic post-processing, and packaging. And finally, the path from a personal problem to a published research tool is shorter than it looks, especially when you start by simply trying to measure something honestly.